# Multimodal Cancer Classification Challenge 2026 — v26 UMBRELLA (v23 + v24 + v25 in one commit)

**One Kaggle commit, three trained models, all derived from v19's proven recipe.**

## What this notebook does

Trains three full-data models **sequentially**, reusing the expensive JPEG cache and pixel
stats across all of them:

| Slot | Sub-version | Backbone | Seed | Purpose |
|---|---|---|---|---|
| 1 | v23 | EfficientNet-B0 | **2** | Seed-diverse copy of v19 (LB 0.7455) for same-arch ensemble |
| 2 | v24 | ResNet-50 | 1 | Teacher's L1a-slide-30 backbone at 128 native (no upscale confounder) |
| 3 | v25 | DenseNet-201 | 1 | Teacher's other L1a-slide-30 backbone at 128 native |

Everything else stays at the v19 recipe: MIL loss, strong aug, AdaBN, 8-way D4 TTA,
test-stain normalization, 12 epochs, AdamW + OneCycleLR, BATCH_SIZE=128, LR=3e-4.

## Why one umbrella

The JPEG cache alone takes ~56 minutes (114k train + 59k test images at ~52 imgs/sec from
the data mount). Running three separate Kaggle commits = ~168 min wasted on cache rebuilds.
Sharing the cache saves nearly 2 hours of pointless I/O.

## Compute on T4 (×2 available, we use 1)

| Stage | Time |
|---|---|
| JPEG cache (one-time, shared) | ~56 min |
| Pixel stats (one-time, shared) | ~30s |
| Train v23 (EffNet-B0, 12 ep) | ~60 min |
| TTA v23 (8-way D4 + AdaBN) | ~7 min |
| Train v24 (ResNet-50, 12 ep) | ~60 min |
| TTA v24 | ~7 min |
| Train v25 (DenseNet-201 mem_eff, 12 ep) | ~90 min |
| TTA v25 | ~8 min |
| Build & write ensembles | <30s |
| **Total** | **~4h 10min** |

Well within Kaggle's 9h ceiling, and uses ~half the compute of three separate commits.

## Outputs (all dumped to `/kaggle/working/`)

| File | Description |
|---|---|
| `submission.csv` | **Auto-submitted** — 3-way rank-average ensemble (safe default) |
| `submission_v23.csv` | EffNet-B0 seed=2 predictions (manual submit on Kaggle UI) |
| `submission_v24.csv` | ResNet-50 predictions (manual submit) |
| `submission_v25.csv` | DenseNet-201 predictions (manual submit) |
| `submission_ensemble_sigmoid.csv` | 3-way sigmoid-average (manual submit) |
| `submission_ensemble_geomean.csv` | 3-way geometric-mean (manual submit) |
| `learning_curves.png` | Per-model train-loss/AUC/time plots |
| `runs/*.pt` | Per-model checkpoints + history.json |

You get **6 candidate submissions** (one auto + five manual) and Kaggle gives you 4/day, so
spread the manual submissions over two days while iterating.

## How to toggle which models run

Edit `MODELS_TO_RUN` in the config cell:
```python
MODELS_TO_RUN = [
    {"name": "v23", "backbone": "efficientnet_b0", "seed": 2},   # ← comment out to skip
    {"name": "v24", "backbone": "resnet50",        "seed": 1},
    {"name": "v25", "backbone": "densenet201",     "seed": 1},
]
```
If v23 already trained in a separate commit, just comment it out — the umbrella will skip
straight to v24 and v25 and save ~67 min.

## IMPORTANT — DO NOT JUST CLICK RUN ALL

To get persistent outputs you must:

1. Click **Save Version** (top-right green button)
2. Choose **Save & Run All (Commit)**
3. Description: `v26 umbrella: v23 + v24 + v25 + 3-way ensemble`
4. **Wait for the commit to finish** (~4h 10min — you'll get a notification)
5. Open the saved version → Output tab → all 6 CSVs are there

Running cells one-by-one in editor mode trains the models but does NOT persist outputs as a Saved Version.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/kaggle/working")

# === Umbrella: which models to train, in order ===
# Each entry trains ONE full-data model. Comment any out to skip.
# Supported backbones: "efficientnet_b0", "resnet50", "densenet201", "resnet18".
MODELS_TO_RUN = [
    {"name": "v23", "backbone": "efficientnet_b0", "seed": 2},
    {"name": "v24", "backbone": "resnet50",        "seed": 1},
    {"name": "v25", "backbone": "densenet201",     "seed": 1},
]

# === Strategy used to populate /kaggle/working/submission.csv (the auto-submitted one) ===
# Options: "rank_avg" (safe default for arch-diverse), "sigmoid_avg", "geomean",
#          or "last" (just the last trained model's predictions).
AUTO_SUBMISSION_STRATEGY = "rank_avg"

# === Shared training recipe (locked to v19, the proven winner) ===
USE_MIL_LOSS        = True   # aux per-patient mean-logit BCE loss
MIL_WEIGHT          = 0.5    # weight of the patient-level loss vs per-cell loss
USE_STRONG_AUG      = True   # ColorJitter(0.4)+RandomErasing+RandomAffine
RANDOM_ERASING_P    = 0.25   # only if USE_STRONG_AUG

USE_TEST_STAIN_NORM = True   # test BF/FL pixel stats
USE_ADABN           = True   # update BN running stats on test before inference
USE_MULTISCALE_TTA  = False  # 8-way D4 only (saves ~6 min inference per model)
TTA_SCALES          = (112, 128, 144)

LABEL_SMOOTHING     = 0.0

# === Training hyperparameters (shared across all models) ===
EPOCHS      = 12
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.0          # disabled (conflicts with MIL grouping)
DROPOUT     = 0.3

# DenseNet-specific safety: gradient checkpointing in dense blocks (same math, ~10% slower fwd,
# but cuts activation memory roughly in half so batch=128 fits on T4 at 128-input).
DENSENET_MEMORY_EFFICIENT = True

NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# v11 hardcoded stats (used when USE_TEST_STAIN_NORM=False)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# Sanity: backbone names valid + uniqueness.
_VALID_BACKBONES = {"efficientnet_b0", "resnet50", "densenet201", "resnet18"}
_seen_names = set()
for m in MODELS_TO_RUN:
    assert m["backbone"] in _VALID_BACKBONES, \
        f"Unknown backbone {m['backbone']!r}, must be one of {_VALID_BACKBONES}"
    assert m["name"] not in _seen_names, f"Duplicate model name {m['name']!r}"
    _seen_names.add(m["name"])
assert len(MODELS_TO_RUN) >= 1, "MODELS_TO_RUN is empty — nothing to do."

print(f"\nConfig (v26 umbrella — {len(MODELS_TO_RUN)} models):")
for m in MODELS_TO_RUN:
    print(f"  {m['name']:>5} | backbone={m['backbone']:>16}  seed={m['seed']}")
print(f"\nShared recipe:")
print(f"  USE_MIL_LOSS        = {USE_MIL_LOSS}  weight={MIL_WEIGHT}")
print(f"  USE_STRONG_AUG      = {USE_STRONG_AUG}  erasing_p={RANDOM_ERASING_P}")
print(f"  USE_TEST_STAIN_NORM = {USE_TEST_STAIN_NORM}")
print(f"  USE_ADABN           = {USE_ADABN}")
print(f"  USE_MULTISCALE_TTA  = {USE_MULTISCALE_TTA}")
print(f"  EPOCHS              = {EPOCHS}  BATCH_SIZE = {BATCH_SIZE}  LR = {LR}")
print(f"  DENSENET_MEMORY_EFFICIENT = {DENSENET_MEMORY_EFFICIENT}")
print(f"  AUTO_SUBMISSION_STRATEGY  = {AUTO_SUBMISSION_STRATEGY!r}")

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v19: also returns patient_id for MIL grouping in training."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
        # Patient ID is -1 for test rows (Name has no pat_X prefix in some splits but in this
        # dataset all names are pat_NN_image_MM.jpg so parse always succeeds).
        self.has_pid = "patient_id" in self.df.columns
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        pid = int(row["patient_id"]) if self.has_pid else -1
        return {"bf": bf, "fl": fl, "label": label, "patient_id": pid, "name": name}

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
# === Backbone builders — one function per supported architecture ===

def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_resnet50_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet50(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features  # 2048
    net.fc = nn.Identity()
    return net, fd

def _make_densenet201_branch(pretrained=True):
    """torchvision densenet201 with 1-ch stem and memory_efficient gradient checkpointing."""
    weights = "DEFAULT" if pretrained else None
    net = models.densenet201(weights=weights, memory_efficient=DENSENET_MEMORY_EFFICIENT)
    old = net.features.conv0
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                         stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.features.conv0 = new_conv
    fd = net.classifier.in_features  # 1920
    net.classifier = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

_BACKBONE_FACTORY = {
    "resnet18":        _make_resnet18_branch,
    "resnet50":        _make_resnet50_branch,
    "densenet201":     _make_densenet201_branch,
    "efficientnet_b0": _make_effnet_b0_branch,
}

def make_branch(backbone, pretrained=True):
    """Public factory used by MultimodalClassifier — dispatches by backbone name string."""
    if backbone not in _BACKBONE_FACTORY:
        raise ValueError(f"Unknown backbone {backbone!r}")
    return _BACKBONE_FACTORY[backbone](pretrained)


class MultimodalClassifier(nn.Module):
    """Dual-branch (BF + FL) classifier with late concat fusion. `backbone` arg selects arch."""
    def __init__(self, backbone, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.backbone_name = backbone
        self.bf_branch, fd = make_branch(backbone, pretrained)
        self.fl_branch, _  = make_branch(backbone, pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)


# Print param counts so we can verify each backbone instantiates and grad-tracks correctly.
print("Sanity check — param counts per requested model:")
with torch.no_grad():
    _x = torch.zeros(2, 1, 128, 128)
    for m in MODELS_TO_RUN:
        _net = MultimodalClassifier(backbone=m["backbone"], pretrained=False).cpu()
        _out = _net(_x, _x)
        _n_params = sum(p.numel() for p in _net.parameters())
        print(f"  {m['name']:>5} | {m['backbone']:>16}  params={_n_params/1e6:.1f}M  out_shape={tuple(_out.shape)}")
        del _net
    del _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

# Compute pixel statistics for stain normalization.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics:")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats (USE_TEST_STAIN_NORM=True)")
else:
    print(f"\nUsing v11 hardcoded normalization (USE_TEST_STAIN_NORM=False)")

In [ ]:
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    """D4 + small rotation, applied identically to BF and FL (paired)."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0,
                 affine_deg=0.0, affine_translate=0.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
        self.affine_deg = affine_deg; self.affine_translate = affine_translate
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        # v19 NEW: paired affine — same translate applied to both modalities so
        # BF/FL stay registered.
        if self.affine_deg > 0 or self.affine_translate > 0:
            H, W = bf.shape[-2], bf.shape[-1]
            angle = random.uniform(-self.affine_deg, self.affine_deg) if self.affine_deg > 0 else 0.0
            tx = random.uniform(-self.affine_translate, self.affine_translate) * W if self.affine_translate > 0 else 0
            ty = random.uniform(-self.affine_translate, self.affine_translate) * H if self.affine_translate > 0 else 0
            bf = TF.affine(bf, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
            fl = TF.affine(fl, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
        return bf, fl

def train_modality_transform(modality):
    """v19: stronger color jitter, with optional RandomErasing applied after normalization."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    if USE_STRONG_AUG:
        steps = [T.ColorJitter(brightness=0.4, contrast=0.4), norm]
        if RANDOM_ERASING_P > 0:
            # RandomErasing operates on normalized tensors; value=0 means it erases to the
            # normalized 0 (which corresponds to original-pixel = mean).
            steps.append(T.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, 0.20),
                                         ratio=(0.3, 3.3), value=0.0))
        return T.Compose(steps)
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

def build_paired_aug():
    """v19 paired aug: D4 + ±10° rot, plus ±15° affine and 10% translate when strong aug is on."""
    if USE_STRONG_AUG:
        return PairedGeoAug(max_rot=10.0, affine_deg=15.0, affine_translate=0.10)
    return PairedGeoAug(max_rot=10.0)

print(f"Augmentation summary:")
print(f"  ColorJitter:      {'brightness/contrast=0.4' if USE_STRONG_AUG else 'brightness/contrast=0.2'}")
print(f"  RandomErasing:    p={RANDOM_ERASING_P if USE_STRONG_AUG else 0.0}")
print(f"  PairedGeoAug:     D4 + ±10° rot" + (" + ±15° affine + 10% translate" if USE_STRONG_AUG else ""))

In [ ]:
def smooth(y, eps):
    if eps <= 0: return y
    return y * (1.0 - eps) + eps * 0.5

def mil_patient_loss(logits, y, patient_ids, pos_weight=None):
    """per-patient mean-logit BCE loss (averages cell-logits within each patient in the batch)."""
    unique_pids = torch.unique(patient_ids)
    if len(unique_pids) < 2:
        return torch.zeros((), device=logits.device, dtype=logits.dtype)
    p_logits, p_labels = [], []
    for pid in unique_pids:
        mask = patient_ids == pid
        p_logits.append(logits[mask].mean())
        p_labels.append(y[mask][0])
    p_logits = torch.stack(p_logits); p_labels = torch.stack(p_labels)
    return F.binary_cross_entropy_with_logits(p_logits, p_labels.float(),
                                              pos_weight=pos_weight)

def run_epoch_train(model, loader, optimizer, scaler, criterion_cell, sched,
                    pos_weight=None, log_every=200):
    model.train()
    losses, hard_ys, ps = [], [], []
    cell_losses, mil_losses = [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf  = batch["bf"].to(DEVICE, non_blocking=True)
        fl  = batch["fl"].to(DEVICE, non_blocking=True)
        y   = batch["label"].float().to(DEVICE, non_blocking=True)
        pid = batch["patient_id"].to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        y_s = smooth(y, LABEL_SMOOTHING)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss_cell = criterion_cell(logits, y_s)
            if USE_MIL_LOSS:
                loss_mil = mil_patient_loss(logits, y, pid, pos_weight=pos_weight)
                loss = loss_cell + MIL_WEIGHT * loss_mil
            else:
                loss_mil = torch.zeros((), device=logits.device)
                loss = loss_cell
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            old_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale() >= old_scale: sched.step()
        else:
            loss.backward()
            if GRAD_CLIP > 0: nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step(); sched.step()
        losses.append(loss.item())
        cell_losses.append(loss_cell.item())
        mil_losses.append(loss_mil.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | "
                  f"total {float(np.mean(losses[-log_every:])):.4f} "
                  f"cell {float(np.mean(cell_losses[-log_every:])):.4f} "
                  f"mil {float(np.mean(mil_losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), float(np.mean(cell_losses)), float(np.mean(mil_losses)), auc


def train_one_model(model_cfg):
    """Train ONE full-data model end-to-end. Returns (ckpt_path, history list)."""
    name     = model_cfg["name"]
    backbone = model_cfg["backbone"]
    seed     = int(model_cfg["seed"])

    print(f"\n=== [{name}] Training full-data model: backbone={backbone}, seed={seed}, "
          f"{EPOCHS} epochs, all {df_train['patient_id'].nunique()} patients ===")
    seed_everything(seed + 100)

    train_ds = CachedCellDataset(df_train, bf_train_cache, fl_train_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=build_paired_aug())
    sampler = PatientBalancedSampler(df_train, batch_size=BATCH_SIZE,
                                     patients_per_batch=PATIENTS_PER_BATCH, seed=seed + 100)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    model = MultimodalClassifier(backbone=backbone, pretrained=True,
                                 dropout=DROPOUT).to(DEVICE)
    pos = (df_train["Diagnosis"] == 1).sum()
    neg = (df_train["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  LR={LR}  "
          f"MIL_W={MIL_WEIGHT if USE_MIL_LOSS else 0.0}  backbone={backbone}")
    criterion_cell = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, steps_per_epoch=len(train_loader),
        epochs=EPOCHS, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    history = []
    for ep in range(EPOCHS):
        t0 = time.time()
        tr_loss, tr_cell, tr_mil, tr_auc = run_epoch_train(
            model, train_loader, optimizer, scaler, criterion_cell, sched,
            pos_weight=pos_weight)
        dt = time.time() - t0
        print(f"  [{name}] ep {ep:>2d} | total {tr_loss:.4f} cell {tr_cell:.4f} "
              f"mil {tr_mil:.4f} tr_auc {tr_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_cell": tr_cell,
                        "tr_mil": tr_mil, "tr_auc": tr_auc, "time": dt})

    ckpt_path = OUT_DIR / f"{name}_fulldata_best.pt"
    torch.save({"model": model.state_dict(), "epoch": EPOCHS - 1,
                "args": {"dropout": DROPOUT, "backbone": backbone, "name": name,
                         "seed": seed}},
               ckpt_path)
    with open(OUT_DIR / f"{name}_history.json", "w") as f:
        json.dump({"name": name, "backbone": backbone, "seed": seed,
                   "history": history}, f, indent=2)
    print(f"  [{name}] saved {ckpt_path}")

    del model, optimizer, sched, scaler, train_loader, train_ds, sampler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return ckpt_path, history


# === Loop over MODELS_TO_RUN ===
trained = []  # list of dicts: {"name", "backbone", "seed", "ckpt", "history"}
overall_t0 = time.time()
for m_cfg in MODELS_TO_RUN:
    ckpt_path, history = train_one_model(m_cfg)
    trained.append({**m_cfg, "ckpt": ckpt_path, "history": history})
    elapsed_min = (time.time() - overall_t0) / 60.0
    print(f"\n[umbrella] cumulative wall-clock since first train = {elapsed_min:.1f} min")

print(f"\n[umbrella] All {len(trained)} models trained successfully.")
for t in trained:
    final_auc = t["history"][-1]["tr_auc"]
    print(f"  {t['name']:>5} | {t['backbone']:>16}  seed={t['seed']}  "
          f"final tr_auc={final_auc:.4f}")

In [ ]:
# Per-model learning curves: train loss components, train AUC, epoch time.
n = len(trained)
fig, axes = plt.subplots(n, 3, figsize=(13, 3.2 * n), squeeze=False)
for row, t in enumerate(trained):
    name, hist = t["name"], t["history"]
    epochs = [e["epoch"] for e in hist]
    ax_l, ax_a, ax_t = axes[row]
    ax_l.plot(epochs, [e["tr_loss"] for e in hist], marker="o", color="tab:blue", label="total")
    ax_l.plot(epochs, [e["tr_cell"] for e in hist], marker="s", color="tab:purple", label="cell BCE")
    ax_l.plot(epochs, [e["tr_mil"]  for e in hist], marker="^", color="tab:orange", label="MIL patient BCE")
    ax_l.legend(); ax_l.set(title=f"[{name}] Train losses", xlabel="epoch", ylabel="loss"); ax_l.grid(True)
    ax_a.plot(epochs, [e["tr_auc"]  for e in hist], marker="o", color="tab:green")
    ax_a.set(title=f"[{name}] Train AUC (cell-level)", xlabel="epoch", ylabel="AUC"); ax_a.grid(True)
    ax_t.plot(epochs, [e["time"]    for e in hist], marker="o", color="tab:red")
    ax_t.set(title=f"[{name}] Epoch time (s)",  xlabel="epoch", ylabel="seconds"); ax_t.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import rankdata

# === AdaBN: update BN running stats on test data before inference ===
@torch.no_grad()
def adabn_pass(model, loader):
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4_at_scale(bf, fl, scale=None):
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model_for_predict(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    backbone = args.get("backbone", "efficientnet_b0")
    model = MultimodalClassifier(backbone=backbone, pretrained=False,
                                 dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model, backbone

def predict_one(ckpt_path, loader, tta_scales=None):
    model, backbone = load_model_for_predict(ckpt_path)
    if USE_ADABN:
        print(f"  AdaBN pass ({backbone})...")
        t_a = time.time()
        adabn_pass(model, loader)
        print(f"  AdaBN done in {time.time()-t_a:.1f}s")
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)


# === Shared test loader (reused across every model's prediction pass) ===
test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

scales_to_use = TTA_SCALES if USE_MULTISCALE_TTA else None
n_aug_total = 8 * (len(TTA_SCALES) if USE_MULTISCALE_TTA else 1)
print(f"Predicting with {n_aug_total}-way TTA (scales={scales_to_use or 'native'}, "
      f"AdaBN={USE_ADABN}) for {len(trained)} models")

# === Per-model TTA passes ===
per_model_preds = {}  # name -> np.ndarray of length len(df_test)
for t in trained:
    name = t["name"]
    print(f"\n--- Predict [{name}] ({t['backbone']}) ---")
    t0 = time.time()
    preds = predict_one(t["ckpt"], test_loader, tta_scales=scales_to_use)
    print(f"  [{name}] inference done in {time.time()-t0:.1f}s  "
          f"mean={preds.mean():.4f}  std={preds.std():.4f}  "
          f"<0.05={(preds<0.05).mean():.2%}  >0.95={(preds>0.95).mean():.2%}")
    per_model_preds[name] = preds
    # Write the per-model CSV.
    pm_path = WORK_DIR / f"submission_{name}.csv"
    pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds}).to_csv(pm_path, index=False)
    print(f"  [{name}] wrote {pm_path}")

# === Build 3 ensemble strategies (equal weights) ===
print(f"\n--- Building ensemble CSVs across {len(per_model_preds)} models ---")
names_in_order = [t["name"] for t in trained]
P = np.stack([per_model_preds[n] for n in names_in_order], axis=0)  # (M, N)
M = P.shape[0]
w = np.ones(M) / M
print(f"  models in ensemble: {names_in_order}  weights={w.tolist()}")

# (1) sigmoid_avg = weighted mean of probs
ens_sigmoid = (P * w[:, None]).sum(axis=0)

# (2) rank_avg = weighted mean of fractional ranks
R = np.zeros_like(P)
for i in range(M):
    R[i] = rankdata(P[i], method="average") / P.shape[1]
ens_rank = (R * w[:, None]).sum(axis=0)

# (3) geomean = exp(weighted mean of log probs, clipped for stability)
eps = 1e-7
ens_geo = np.exp((np.log(np.clip(P, eps, 1 - eps)) * w[:, None]).sum(axis=0))

ensembles = {
    "sigmoid":  ens_sigmoid,
    "rank_avg": ens_rank,
    "geomean":  ens_geo,
}
# Per-strategy CSVs (manual submit on Kaggle UI).
for strat, preds in ensembles.items():
    p = WORK_DIR / f"submission_ensemble_{strat}.csv"
    pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds}).to_csv(p, index=False)
    print(f"  wrote {p}  mean={preds.mean():.4f}  std={preds.std():.4f}")

# === Pick which one becomes /kaggle/working/submission.csv (auto-submitted) ===
strat = AUTO_SUBMISSION_STRATEGY
if strat == "last":
    auto_preds = per_model_preds[names_in_order[-1]]
    auto_label = f"last model ({names_in_order[-1]})"
elif strat == "rank_avg":
    auto_preds = ens_rank
    auto_label = "ensemble rank_avg"
elif strat == "sigmoid_avg":
    auto_preds = ens_sigmoid
    auto_label = "ensemble sigmoid_avg"
elif strat == "geomean":
    auto_preds = ens_geo
    auto_label = "ensemble geomean"
else:
    raise ValueError(f"Unknown AUTO_SUBMISSION_STRATEGY {strat!r}")

pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": auto_preds}).to_csv(
    WORK_DIR / "submission.csv", index=False)
print(f"\nWrote /kaggle/working/submission.csv  ({auto_label})  "
      f"mean={auto_preds.mean():.4f}  min={auto_preds.min():.4f}  max={auto_preds.max():.4f}")

# Summary table.
print(f"\n=== Output summary ===")
all_csvs = sorted(WORK_DIR.glob("submission*.csv"))
for f in all_csvs:
    arr = pd.read_csv(f)["Diagnosis"].values
    print(f"  {f.name:>40}  rows={len(arr):>6}  mean={arr.mean():.4f}  std={arr.std():.4f}")

!wc -l /kaggle/working/submission*.csv